# Lab F — Fine-tuning: earn it, then fine-tune the retriever

**Curriculum §5 · Track 3**

**The rule: you may not fine-tune until prompting is exhausted and measured.** This lab does the highest-ROI, genuinely-free fine-tune for a RAG system — **fine-tuning the embedding model** on your own corpus — and proves the gain on the harness. (LLM QLoRA with Unsloth is the alternative when you need *behaviour/format* change; embedding-FT is what moves *retrieval*.)

## ▶ Colab setup — run this cell first

1. Add your keys in Colab: **🔑 (left sidebar) → Secrets** → add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `GROQ_API_KEY` (toggle *Notebook access* on).
2. Free signups: **Langfuse** → cloud.langfuse.com · **Groq** → console.groq.com
3. Run the cell. It installs deps and clones the shared `common/` package from GitHub.

> **If you see a `PIL._typing._Ink` import error:** run the cell, then **Runtime → Restart session**, then re-run. It's a Colab package clash, fixed by the Pillow upgrade + a restart.

In [ ]:
# --- Colab bootstrap (safe to re-run) ---
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run('pip install -q -U Pillow'.split())          # fix Colab PIL._typing._Ink clash
    subprocess.run('pip install -q langfuse ragas sentence-transformers faiss-cpu rank_bm25 langchain langchain-community langchain-groq langchain-text-splitters langgraph pypdf pdfplumber PyMuPDF pandas'.split())

    # The repo is public — clone it to get the shared common/ package.
    REPO = pathlib.Path('/content/labpractice')
    if not (REPO/'common'/'harness.py').exists():
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/AICareerstack26/labpractice', str(REPO)])
    sys.path.insert(0, str(REPO))

    # Keys from Colab Secrets (🔑 sidebar) -> env vars that common/config.py reads.
    try:
        from google.colab import userdata
        for k in ['LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY','GROQ_API_KEY']:
            v = userdata.get(k)
            if v: os.environ[k] = v
        os.environ.setdefault('LANGFUSE_HOST','https://cloud.langfuse.com')
    except Exception as e:
        print('Secrets not set — running offline. (', e, ')')
else:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))          # local fallback

print('Environment:', 'Colab' if IN_COLAB else 'Local')

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parent))              # make `common` importable locally
from common.config import *
from common.corpus import DOCS, current_docs, SUPERSEDED_IDS
from common.golden import GOLDEN
from common.obs import observe, span_meta, trace_meta, make_config, flush, ENABLED
from common.harness import evaluate, leaderboard
print('Langfuse tracing:', 'ON' if ENABLED else 'OFF (labs still run)')
print('AS_OF:', AS_OF, '| in force:', [d['id'] for d in current_docs()], '| superseded:', SUPERSEDED_IDS)

## 1 · F0 — the number fine-tuning must beat

Baseline retrieval quality with the off-the-shelf `bge-small` embedder. Write it down.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

split_docs = current_docs()
CORPUS = [dict(id=d['id'], text=f"[{d['doc']} {d['version']}] {d['text']}") for d in split_docs]

def retrieval_scores(model):
    M = model.encode([c['text'] for c in CORPUS], normalize_embeddings=True)
    hit = mrr = n = 0
    for g in GOLDEN:
        if g['source'] is None: continue
        n += 1
        qv = model.encode([g['q']], normalize_embeddings=True)[0]
        order = list(np.argsort(-(M @ qv)))
        ids = [CORPUS[i]['id'] for i in order]
        if g['source'] in ids[:3]: hit += 1
        if g['source'] in ids: mrr += 1.0/(ids.index(g['source'])+1)
    return dict(hit_at_3=round(hit/n,3), mrr=round(mrr/n,3))

base = SentenceTransformer(EMBED_MODELS['small'])
print('BASE :', retrieval_scores(base))

## 2 · Build leakage-free training pairs

Positives = (question, the chunk that truly answers it). `MultipleNegativesRankingLoss` uses the *other* positives in the batch as negatives — no manual negatives needed. Tiny data here is illustrative; a real run wants hundreds of pairs and a held-out split.

In [ ]:
by_id = {c['id']: c['text'] for c in CORPUS}
pairs = [InputExample(texts=[g['q'], by_id[g['source']]])
         for g in GOLDEN if g['source'] in by_id]
print(len(pairs), 'training pairs')
loader = DataLoader(pairs, shuffle=True, batch_size=4)

## 3 · Fine-tune (a few epochs — seconds on a T4)

In [ ]:
ft = SentenceTransformer(EMBED_MODELS['small'])
loss = losses.MultipleNegativesRankingLoss(ft)
ft.fit(train_objectives=[(loader, loss)], epochs=8, warmup_steps=2, show_progress_bar=True)
print('BASE       :', retrieval_scores(base))
print('FINE-TUNED :', retrieval_scores(ft))

## 4 · Prove it end-to-end on the harness

Retrieval scores are the leading indicator; now run the *full* pipeline with each embedder through `evaluate`.

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model=GEN_MODEL, temperature=0); judge = ChatGroq(model=JUDGE_MODEL, temperature=0)
ACTIVE = {'model': base}
PROMPT = ("You are Meridian Bank's copilot. Answer ONLY from context, cite [ids], else INSUFFICIENT_CONTEXT."
          "\n\nContext:\n{ctx}\n\nQuestion: {q}")

@observe(name='rag.request')
def pipeline(query, cfg):
    m = ACTIVE['model']
    M = m.encode([c['text'] for c in CORPUS], normalize_embeddings=True)
    qv = m.encode([query], normalize_embeddings=True)[0]
    idx = np.argsort(-(M @ qv))[:cfg['k']]
    hits = [dict(CORPUS[i]) for i in idx]
    ctx = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    ans = llm.invoke(PROMPT.format(ctx=ctx, q=query)).content
    span_meta(embedder=cfg['embed_model']); trace_meta(tags=[cfg['hash']], config=cfg)
    return dict(answer=ans, hits=hits)

rows = []
for name, model in [('bge-small (base)', base), ('bge-small (fine-tuned)', ft)]:
    ACTIVE['model'] = model
    rows.append(evaluate(make_config(embed_model=name, k=3), pipeline, judge=judge))
leaderboard(rows, sort_by='mrr')[['config','embed_model','hit_at_k','mrr','version_correct','faithfulness']]

## 5 · What you should conclude

- Fine-tuning the **embedder** on your own corpus is cheap, fast, and moves `hit_at_k`/`mrr` — the retrieval quality that caps everything downstream. This is the fine-tune with the best ROI for RAG.
- **The central discovery:** fine-tuning reliably changes *behaviour, format, and tone*; it is an expensive, unreliable way to add *knowledge* — that is RAG's job. If your failure is 'wrong facts,' fix retrieval, not weights.
- Only reach for **LLM QLoRA (Unsloth/PEFT)** once prompting + RAG are exhausted and you need a specific *format* or *tone* the base model won't hold.

**Next →** `lab06` (agentic layer)